In [1]:
# import sys
# !{sys.executable} -m pip install -U plotly

In [2]:
import kaleido
import plotly.io as pio

In [3]:
import os
import glob
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.stats as sp
import scikit_posthocs as sp_post


In [4]:
# configuration
LOG_DIR = './log_mo'       # where logs are stored
OUTPUT_DIR = './plots_mo'  #where I'll save the plots

COLORS = {
    'Train_RMSE': '#1f77b4',
    'Test_RMSE': '#ff7f0e',
    'Std_Dev': '#9467bd',
    'Size': '#2ca02c',
    'Features': '#d62728',
    'NAO': '#17becf',
    'NAOC': '#e377c2'
}

OBJ_MAP = {
    0: "RMSE",
    1: "Size",
    2: "Features",
    3: "NAO",
    4: "NAOC"
}

DATASETS = ['Toxicity']

In [5]:
# Transforms a column with "5.1|10.2" strings into a DataFrame of floats
def parse_fitness_column(series, log_file_name="Unknown", allow_na=False):
    # deal with NaN values first
    if series.isna().any():
        if allow_na:
            # if it is allowed, return None
            return None
        else:
            # if it is mandatory (like RMSE), NaN is a critical error
            raise ValueError(
                f"[CRITICAL ERROR] Found NaN in mandatory column in file '{log_file_name}'. "
                "This means the Elite was not evaluated correctly."
            )

    # Convert to string for uniform processing
    series_str = series.astype(str)

    # Handle the literal string "N/A" (in case pandas did not convert it to NaN)
    if (series_str == "N/A").any():
        if allow_na:
            return None
        else:
            error_idx = series_str[series_str == "N/A"].index[0]
            raise ValueError(
                f"[CRITICAL ERROR] Found 'N/A' string in file '{log_file_name}' at line {error_idx}."
            )

    # try to split and convert to float
    try:
        return series_str.str.split('|', expand=True).astype(float)
    except ValueError as e:
        raise ValueError(f"[CRITICAL] Could not convert fitness to numbers in {log_file_name}. Error: {e}")

In [6]:
# Function to load data from all logs
def load_data_for_analysis(dataset_name, objectives_key, view_mode='main'):
    target_path = os.path.join(LOG_DIR, dataset_name)
    
    if not os.path.exists(target_path):
        print(f"[ERROR] Folder not found: {target_path}")
        return None, None

    # Find all scenarios
    scenarios = [d for d in os.listdir(target_path) if os.path.isdir(os.path.join(target_path, d))]
    
    history_data = {} 
    final_rows = []   

    print(f"Logs from {dataset_name} ({objectives_key}) - View: {view_mode.upper()}")

    for scen in scenarios:
        pattern = os.path.join(target_path, scen, objectives_key, "fold_*", "execution_log.csv")
        files = glob.glob(pattern)
        
        if not files: continue
        
        history_data[scen] = []
        
        for f in files:
            try:
                df = pd.read_csv(f)
                
                # --- INDEX MAPPING (Based on MOGP Logger) ---                
                # 0: Algo, 1: UUID, 2: Dataset, 3: Seed
                # 4: Gen, 5: Main Train, 6: Time, 7: Main Nodes
                # 8: Main Test Fit
                # 9: RMSE Elite TRAIN Fit
                # 10: RMSE Elite TEST Fit
                # 11: RMSE Elite Nodes
                # 12: Std Dev
                # 13: Ideal Point

                idx_offset = 4 
                
                gen_idx = 0 + idx_offset  # 4
                std_idx = 8 + idx_offset  # 12
                ideal_idx = 9 + idx_offset # 13               

                if view_mode == 'rmse':
                    # RMSE View: Main Train and RMSE Elite Test
                    train_idx = 5 + idx_offset #9
                    test_idx = 6 + idx_offset  #10
                    size_idx = 7 + idx_offset  #11
                else:
                    # Main View: Main Train + Main Test
                    train_idx = 1 + idx_offset
                    test_idx = 4 + idx_offset
                    size_idx = 3 + idx_offset
                
                # Parse Fitness Columns    
                train_fits = parse_fitness_column(df.iloc[:, train_idx], f, allow_na=False)
                test_fits = parse_fitness_column(df.iloc[:, test_idx], f, allow_na=False)
                
                # Ideal Point (allow_na=True because it is N/A for most scenarios)
                ideal_fits = parse_fitness_column(df.iloc[:, ideal_idx], f, allow_na=True)

                # Clean DataFrame
                clean_df = pd.DataFrame({
                    'Generation': df.iloc[:, gen_idx],
                    'Std_RMSE': df.iloc[:, std_idx]
                })
                
                # Add Train Metrics
                if train_fits is not None:
                    for c in train_fits.columns:
                        clean_df[f'Train_{OBJ_MAP.get(c, c)}'] = train_fits[c]
                
                # Add Test Metrics
                if test_fits is not None:
                    for c in test_fits.columns:
                        clean_df[f'Test_{OBJ_MAP.get(c, c)}'] = test_fits[c]
                
                clean_df['Test_Size'] = df.iloc[:, size_idx]
                
                # Add Ideal Point if available
                if ideal_fits is not None:
                    clean_df['Ideal_RMSE'] = ideal_fits[0]

                history_data[scen].append(clean_df)
                
                # Data for Boxplots (Last Generation)
                last_row = clean_df.iloc[-1]
                try: fold_num = int(f.split(os.sep)[-2].split('_')[-1])
                except: fold_num = 0

                row_dict = {'Dataset': dataset_name, 'Scenario': scen, 'Fold': fold_num}
                for col in clean_df.columns:
                    if col.startswith('Train_') or col.startswith('Test_'):
                        row_dict[col] = last_row[col]
                final_rows.append(row_dict)
            except Exception as e:
                print(f"  [WARN] Could not process file {f}: {e}")

    final_df = pd.DataFrame(final_rows)
    return history_data, final_df

In [7]:
def plot_convergence(dataset_name, objectives_key, history_data, view_mode):
    save_dir = os.path.join(OUTPUT_DIR, dataset_name, objectives_key)
    if not os.path.exists(save_dir): os.makedirs(save_dir)

    for scen, folds_list in history_data.items():
        if not folds_list: continue

        sample_df = folds_list[0]
        metrics = [c.replace("Test_", "") for c in sample_df.columns if c.startswith("Test_")]
        min_len = min([len(df) for df in folds_list])
        generations = np.arange(min_len)

        fig = make_subplots(
            rows=1, cols=3,
            subplot_titles=("Performance (RMSE)", "Structural Complexity", "Pop RMSE Std Dev"),
            horizontal_spacing=0.08
        )

        def add_trace(col, name, color, idx, show_leg=True):
            if col not in sample_df.columns: return

            data = np.array([df[col].values[:min_len] for df in folds_list])
            median = np.median(data, axis=0)
            q1 = np.percentile(data, 15, axis=0)
            q3 = np.percentile(data, 85, axis=0)

            # IQR shading
            fig.add_trace(go.Scatter(
                x=np.concatenate([generations, generations[::-1]]),
                y=np.concatenate([q3, q1[::-1]]),
                fill='toself', fillcolor=color, opacity=0.15,
                line=dict(width=0), showlegend=False), row=1, col=idx)

            # Median Line
            fig.add_trace(go.Scatter(
                x=generations, y=median, mode='lines', name=name,
                line=dict(color=color)), row=1, col=idx)

        # Plot 1: Performance
        add_trace('Train_RMSE', 'Train RMSE', COLORS['Train_RMSE'], 1)
        add_trace('Test_RMSE', 'Test RMSE', COLORS['Test_RMSE'], 1)
        
        # Add Ideal Point line if it exists
        if 'Ideal_RMSE' in sample_df.columns:
             add_trace('Ideal_RMSE', 'Ideal Point', 'black', 1, show_leg=True)

        # Plot 2: Complexity
        add_trace('Test_Size', 'Size', COLORS['Size'], 2)
        for metric in [m for m in metrics if m != "RMSE" and m != "Size"]:
            color = COLORS.get(metric, '#333333')
            add_trace(f'Test_{metric}', metric, color, 2)

        # Plot 3: Std Dev
        add_trace('Std_RMSE', 'RMSE Std Dev', COLORS['Std_Dev'], 3, show_leg=False)

        fig.update_layout(
            title=f"Convergence Analysis: {scen} ({dataset_name} | {objectives_key} | {view_mode.upper()})",
            height=500, width=1400, template="plotly_white",
            legend=dict(orientation="h", y=-0.15, x=0.5, xanchor="center")
        )
        
        file_path = os.path.join(save_dir, f"convergence_{scen}_VIEW_{view_mode.upper()}.png")
        fig.write_image(file_path)
        print(f"  -> Saved convergence: {file_path}")

In [8]:
def plot_boxplots(dataset_name, objectives_key, final_df, view_mode):
    save_dir = os.path.join(OUTPUT_DIR, dataset_name, objectives_key)
    
    if final_df.empty: return

    df_melt = final_df.melt(
        id_vars=['Scenario'], 
        value_vars=['Train_RMSE', 'Test_RMSE'],
        var_name='Type', value_name='RMSE'
    )
    
    plt.figure(figsize=(12, 7))
    sns.set_style("whitegrid")

    sns.boxplot(
        data=df_melt, x='Scenario', y='RMSE', hue='Type',
        palette=[COLORS['Train_RMSE'], COLORS['Test_RMSE']],
        width=0.6, linewidth=1.2, showfliers=False 
    )
    sns.stripplot(
        data=df_melt, x='Scenario', y='RMSE', hue='Type',
        dodge=True, color='black', alpha=0.3, size=3, legend=False
    )

    plt.title(f'Final RMSE Distribution: {dataset_name} ({view_mode.upper()})', fontsize=14)
    plt.xticks(rotation=45)
    plt.tight_layout()
    
    filename = f"boxplot_rmse_VIEW_{view_mode.upper()}.png"
    plt.savefig(os.path.join(save_dir, filename), dpi=300)
    plt.close()
    print(f"  -> Saved RMSE boxplot.")

In [9]:
def run_stats(dataset_name, objectives_key, final_df, view_mode, save_dir):
    if final_df.empty: return
    
    try:
        pivot_df = final_df.pivot(index='Fold', columns='Scenario', values='Test_RMSE')
        
        output_txt = [f"=== Stats: {dataset_name} ({objectives_key}) ===\n"]
        output_txt.append(pivot_df.describe().to_string() + "\n\n")
        
        stat, p = sp.friedmanchisquare(*[pivot_df[col] for col in pivot_df.columns])
        output_txt.append(f"Friedman Test:\nStatistic: {stat:.4f}, p-value: {p:.6f}\n")
        
        if p < 0.05:
            output_txt.append("\nSignificant differences found. Nemenyi Post-Hoc:\n")
            nemenyi = sp_post.posthoc_nemenyi_friedman(pivot_df)
            output_txt.append(nemenyi.to_string())
        
        filename = f"stats_VIEW_{view_mode.upper()}.txt"
        with open(os.path.join(save_dir, filename), "w") as f:
            f.write("".join(output_txt))
            
    except Exception as e:
        print(f"Stats Error: {e}")

In [10]:
def generate_all_plots(dataset_name, objectives_key='2_Objs', view_mode='main'):
    print(f"\n--- Processing {dataset_name} [{objectives_key}] ---")
    
    history_data, final_df = load_data_for_analysis(dataset_name, objectives_key, view_mode)
    
    if final_df is None or final_df.empty:
        print("No data found.")
        return

    plot_convergence(dataset_name, objectives_key, history_data, view_mode)
    plot_boxplots(dataset_name, objectives_key, final_df, view_mode)
    
    save_dir = os.path.join(OUTPUT_DIR, dataset_name, objectives_key)
    run_stats(dataset_name, objectives_key, final_df, view_mode, save_dir)



In [11]:
for ds in DATASETS: # DATASETS dict defined in run_experiments, or hardcode list here
    for obj in ['2_Objs', '3_Objs', '5_Objs']:
        for mode in ['main', 'rmse']:
            generate_all_plots(ds, obj, mode)


--- Processing Toxicity [2_Objs] ---
Logs from Toxicity (2_Objs) - View: MAIN
  -> Saved convergence: ./plots_mo/Toxicity/2_Objs/convergence_NT_NSGA-II_VIEW_MAIN.png
  -> Saved convergence: ./plots_mo/Toxicity/2_Objs/convergence_NT_1st_Obj_Elitism_VIEW_MAIN.png
  -> Saved convergence: ./plots_mo/Toxicity/2_Objs/convergence_NSGA-II_Pure_VIEW_MAIN.png


/var/folders/pw/2p3gn8w505d3kt196c0t1_x40000gn/T/ipykernel_26764/3295342174.py:20: FutureWarning:



Setting a gradient palette using color= is deprecated and will be removed in v0.14.0. Set `palette='dark:black'` for the same effect.




  -> Saved RMSE boxplot.

--- Processing Toxicity [2_Objs] ---
Logs from Toxicity (2_Objs) - View: RMSE
  -> Saved convergence: ./plots_mo/Toxicity/2_Objs/convergence_NT_NSGA-II_VIEW_RMSE.png
  -> Saved convergence: ./plots_mo/Toxicity/2_Objs/convergence_NT_1st_Obj_Elitism_VIEW_RMSE.png
  -> Saved convergence: ./plots_mo/Toxicity/2_Objs/convergence_NSGA-II_Pure_VIEW_RMSE.png


/var/folders/pw/2p3gn8w505d3kt196c0t1_x40000gn/T/ipykernel_26764/3295342174.py:20: FutureWarning:



Setting a gradient palette using color= is deprecated and will be removed in v0.14.0. Set `palette='dark:black'` for the same effect.




  -> Saved RMSE boxplot.

--- Processing Toxicity [3_Objs] ---
Logs from Toxicity (3_Objs) - View: MAIN
  -> Saved convergence: ./plots_mo/Toxicity/3_Objs/convergence_NT_NSGA-II_VIEW_MAIN.png
  -> Saved convergence: ./plots_mo/Toxicity/3_Objs/convergence_NT_1st_Obj_Elitism_VIEW_MAIN.png
  -> Saved convergence: ./plots_mo/Toxicity/3_Objs/convergence_NSGA-II_Pure_VIEW_MAIN.png


...

  -> Saved RMSE boxplot.

--- Processing Toxicity [5_Objs] ---
Logs from Toxicity (5_Objs) - View: MAIN
  -> Saved convergence: ./plots_mo/Toxicity/5_Objs/convergence_NT_NSGA-II_VIEW_MAIN.png
  -> Saved convergence: ./plots_mo/Toxicity/5_Objs/convergence_NT_1st_Obj_Elitism_VIEW_MAIN.png
  -> Saved convergence: ./plots_mo/Toxicity/5_Objs/convergence_NSGA-II_Pure_VIEW_MAIN.png


/var/folders/pw/2p3gn8w505d3kt196c0t1_x40000gn/T/ipykernel_26764/3295342174.py:20: FutureWarning:



Setting a gradient palette using color= is deprecated and will be removed in v0.14.0. Set `palette='dark:black'` for the same effect.




  -> Saved RMSE boxplot.

--- Processing Toxicity [5_Objs] ---
Logs from Toxicity (5_Objs) - View: RMSE
  -> Saved convergence: ./plots_mo/Toxicity/5_Objs/convergence_NT_NSGA-II_VIEW_RMSE.png
  -> Saved convergence: ./plots_mo/Toxicity/5_Objs/convergence_NT_1st_Obj_Elitism_VIEW_RMSE.png
  -> Saved convergence: ./plots_mo/Toxicity/5_Objs/convergence_NSGA-II_Pure_VIEW_RMSE.png


/var/folders/pw/2p3gn8w505d3kt196c0t1_x40000gn/T/ipykernel_26764/3295342174.py:20: FutureWarning:



Setting a gradient palette using color= is deprecated and will be removed in v0.14.0. Set `palette='dark:black'` for the same effect.




  -> Saved RMSE boxplot.
